## Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [4]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3.8-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0'}}, client=<groq.resources.chat.completions.Completions object at 0x000001CE054ED2A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001CE054ED1B0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [1]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:str=Field(description="The year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:str=Field(description="The rating of the movie out of 10")

In [5]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0'}}, client=<groq.resources.chat.completions.Completions object at 0x000001CE054ED2A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001CE054ED1B0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'string'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The rating of the movie out of 10', 'type': 'string'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 

In [4]:
model.invoke("Provide the details about the movie Inception")

AIMessage(content='**Inception** is a landmark science fiction heist film directed by and written by **Christopher Nolan**, released in **2010**. It is widely regarded as one of the most complex and influential blockbusters of the 21st century.\n\n### Key Details\n\n- **Title:** Inception\n- **Release Year:** 2010\n- **Director:** Christopher Nolan\n- **Writers:** Christopher Nolan\n- **Starring:**\n  - Leonardo DiCaprio as **Dom Cobb**\n  - Joseph Gordon-Levitt as **Arthur**\n  - Elliot Page as **Ariadne**\n  - Tom Hardy as **Eames**\n  - Ken Watanabe as **Saito**\n  - Marion Cotillard as **Mal Cobb**\n  - Cillian Murphy as **Robert Fischer**\n  - Michael Caine as **Miles**\n  - Tom Berenger as **Bryce**\n- **Runtime:** 148 minutes\n- **Genre:** Science Fiction, Action, Thriller, Drama\n- **Score:** Hans Zimmer (featuring the iconic track *"Time"*)\n\n### Plot Summary\n\nThe film follows **Dom Cobb** (Leonardo DiCaprio), a skilled thief who specializes in "inception"—the act of planti

In [7]:
model_with_structure.invoke("Provide the details of the Movie Inception")

Movie(title='Inception', year='2010', director='Christopher Nolan', rating='8.8')

### Message output alonside parsed structure

In [13]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'k2h6xp7yj', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 355, 'total_tokens': 423, 'completion_time': 0.171561156, 'completion_tokens_details': None, 'prompt_time': 0.025661358, 'prompt_tokens_details': None, 'queue_time': 0.193291371, 'total_time': 0.197222514}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0af3c-5a35-7f10-977d-62b73e98b488-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.8, 'title': 'Inception', 'year': 2010}, 'id': 'k2h6xp7yj', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 355, 'output_tokens': 68, 'total

In [14]:
### Nested Structure
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal Cobb'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

## TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [8]:
from typing_extensions import TypedDict,Annotated
# field: Annotated[data_type, default_value, description]
class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Provide details for the 2012 movie The Avengers. "
    "Use its IMDb rating for the rating field.")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [17]:
# Nested TypedDict 
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: Annotated[float | None, ..., "Budget in millions USD"]

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 356,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Paul Rudd', 'role': 'Scott Lang / Ant-Man'},
  {'name': 'Brie Larson', 'role': 'Carol Danvers / Captain Marvel'},
  {'name': 'Josh Brolin', 'role': 'Thanos'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'Avengers: Endgame',
 'year': 2019}

In [23]:
print(model.profile)

None


## DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [24]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [25]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')